# 03 · DentVoxel 3D CBCT WSP Benchmark (Step-by-Step Validation & Visuals)

Self-contained, resume-safe 3D CBCT benchmark using `DentVoxel.zip` (100 calibrated-HU NIfTI scans: **60 train / 20 val / 20 sealed test**).

**Run cleanly from top to bottom on a Colab GPU (T4 / L4):**
- **Step 1:** Mount Google Drive, locate `dental-wsp` & `DentVoxel.zip`, verify GPU + calibrated HU (`[-1000, +3000]`), and lock the deterministic 60/20/20 scan split on Drive.
- **Step 2 (`dentvoxel_cls` · Stage 1):** Frozen WSP (`K = 0`, 6 replies) on **Scan-Split Axial Maxillary Sinus Presence** across the 20 validation scans.
- **Step 3 (`dentvoxel_cls` · Stages 2 & 3):** 3-Arm learning curve (`K = 2..60` scans, 5 seeds, Drive-checkpointed).
- **Step 4 (`dentvoxel_seg` · Stage 1):** Frozen 3D `cbct-wsp82-v1` (82-param 7-tap 3D cross-stencil) full 3D mechanistic audit on the 20 validation CBCT scans (3D swaps, outside FP share, 3D surface recall, **3D excluded tooth-voxel rate**).
- **Step 5 (`dentvoxel_seg` · Stages 2 & 3):** Converged 4-Arm 3D 82-param learning curve (`K = 1..60` scans, 3 seeds, `batch_size=4`, `epochs=120`, `lr=3e-2`, cosine LR, Drive-checkpointed after every step).
- **Step 6 (Visuals · Frozen Replies):** Deterministic **Axial & Coronal** mid-arch CBCT slice overlay grids across all 6 frozen replies.
- **Step 7 (Visuals · 4 Arms):** Side-by-side Axial & Coronal CBCT comparison across all 4 arms at `K = 60`.

**Governance:** The 20 held-out test scans stay sealed (`split='val'`). Clear outputs before committing to Git.

## Step 1 · Setup, Drive Mount, HU Audit & Deterministic 60/20/20 Scan Split

In [ ]:
from pathlib import Path
import json
import sys

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_SRC = Path("/content/drive/MyDrive/WSP_Dental_Data/DentVoxel.zip")
    candidates = [Path("/content/dental-wsp"), Path("/content/drive/MyDrive/dental-wsp")]
    REPO_DIR = next((p for p in candidates if (p / "benchmarks").is_dir()), candidates[1])
    OUT_ROOT = Path("/content/drive/MyDrive/dental-wsp-runs/dentvoxel")
else:
    REPO_DIR = Path(r"C:\Users\Lenovo\Desktop\dental-wsp")
    DATA_SRC = Path(r"C:\Users\Lenovo\Desktop\WSP_Dental_Data\DentVoxel.zip")
    OUT_ROOT = REPO_DIR / "runs/dentvoxel"

assert (REPO_DIR / "benchmarks").is_dir(), f"dental-wsp not found at {REPO_DIR}"
assert DATA_SRC.exists(), f"DentVoxel ZIP not found at {DATA_SRC}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

OUT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from benchmarks.dentvoxel.adapter import make_dentvoxel_scan_splits, verify_dentvoxel_directory
from wsp_core.guards import exposure_guard
exposure_guard(DATA_SRC.parent)
audit = verify_dentvoxel_directory(DATA_SRC, max_scans=3)
print(f"Repo:        {REPO_DIR}")
print(f"Data:        {DATA_SRC}")
print(f"Checkpoints: {OUT_ROOT}")
print(f"Device:      {DEVICE}")
print(json.dumps(audit, indent=2))

SPLIT_FILE = OUT_ROOT / "dentvoxel_splits.private.json"
if not SPLIT_FILE.is_file():
    scan_ids = [f"scan_{i:04d}" for i in range(1, 101)]
    SPLIT_FILE.write_text(json.dumps(make_dentvoxel_scan_splits(scan_ids, seed=2026), indent=2), encoding="utf-8")
print(f"Deterministic 60/20/20 scan split locked at {SPLIT_FILE}")

## Step 2 · `dentvoxel_cls` Stage 1 — Frozen WSP (`K = 0`, Axial Maxillary Sinus Presence, 6 Replies)

In [ ]:
import pandas as pd
from benchmarks.dentvoxel.adapter import load_dentvoxel_cls
from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.schemas import strict_json_loads, verify_response_lock

print("Loading 60 train + 20 val scans for scan-split axial maxillary sinus slices (takes ~1-2 min from ZIP)...")
train_cls = load_dentvoxel_cls(DATA_SRC, SPLIT_FILE, split="train", norm_mode="hu_window", stride=4)
val_cls = load_dentvoxel_cls(DATA_SRC, SPLIT_FILE, split="val", norm_mode="hu_window", stride=4)
print(f"dentvoxel_cls slices: train={len(train_cls)} (pos={sum(s.label for s in train_cls)}), val={len(val_cls)} (pos={sum(s.label for s in val_cls)})")

cls_design = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_cls.design.lock.json"
cls_resp = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_cls.responses.lock.json"
cls_d_bytes = cls_design.read_bytes()
frozen_cls = verify_response_lock(strict_json_loads(cls_d_bytes), cls_d_bytes, strict_json_loads(cls_resp.read_bytes()), "cls-lenet2d-v1")

cls_stage1_df = pd.DataFrame([{"Reply": slot, **evaluate_classifier(p, val_cls)} for slot, p in frozen_cls.items()]).set_index("Reply")
cls_stage1_df.to_csv(OUT_ROOT / "dentvoxel_cls_stage1_frozen.csv")
display(cls_stage1_df[["roc_auc", "pr_auc", "mean_within_group_roc_auc", "accuracy", "margin_ablated_roc_auc", "sign_flip_flagged"]].round(4))

## Step 3 · `dentvoxel_cls` Stages 2 & 3 — 3-Arm Scan-Budget Learning Curve (`K = 2..60` Scans, 5 Seeds)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.training.runner import plot_classification_learning_curve, run_classification_three_arm_validation

CLS_OUT = OUT_ROOT / "dentvoxel_cls"
cls_runs, cls_summary = run_classification_three_arm_validation(
    task="dentvoxel_cls",
    train_samples=train_cls,
    val_samples=val_cls,
    design_lock_path=cls_design,
    responses_lock_path=cls_resp,
    budgets=[2, 4, 8, 16, 32, 60],
    seeds=[0, 1, 2, 3, 4],
    epochs=200,
    lr=5e-3,
    device=DEVICE,
    out_dir=CLS_OUT,
    resume=True,
)
display(cls_summary.round(4))
fig = plot_classification_learning_curve(cls_summary, task="dentvoxel_cls")
plt.show()
plt.close(fig)
print(f"Saved dentvoxel_cls runs, summary CSV, and PNG to {CLS_OUT}")

## Step 4 · `dentvoxel_seg` Stage 1 — Frozen 3D WSP Mechanistic Audit (`K = 0`, 82-Param 3D Stencil, 20 Val Scans)

In [ ]:
import pandas as pd
from benchmarks.dentvoxel.adapter import load_dentvoxel_seg
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.schemas import strict_json_loads, verify_response_lock

print("Loading & resampling 20 validation 3D CBCT scans to 96x128x128...")
val_seg = load_dentvoxel_seg(DATA_SRC, SPLIT_FILE, split="val", norm_mode="hu_window")
seg_design = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_seg.design.lock.json"
seg_resp = REPO_DIR / "benchmarks/dentvoxel/locks/dentvoxel_seg.responses.lock.json"
seg_d_bytes = seg_design.read_bytes()
frozen_seg = verify_response_lock(strict_json_loads(seg_d_bytes), seg_d_bytes, strict_json_loads(seg_resp.read_bytes()), "cbct-wsp82-v1")

seg_rows = []
for slot, payload in frozen_seg.items():
    sw = evaluate_segmentation_swaps(payload, val_seg)
    au = evaluate_segmentation_paired_audit(payload, val_seg)
    seg_rows.append({
        "Reply": slot,
        "Intact 3D Dice (%)": sw["full_cohort_dice_a_pct"],
        "Image-only 3D Dice (%)": sw["full_cohort_dice_b_pct"],
        "Position-only 3D Dice (%)": sw["full_cohort_dice_c_pct"],
        "3D Swap loss A (pp)": sw["intact"]["swap_loss_pp"],
        "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
        "Delta 3D Surface F1 (pp)": au["bf1_gain_pp"],
        "Delta 3D Surface Recall (pp)": au["edge_recall_gain_pp"],
        "3D Box IoU A (%)": au["box_iou_a_pct"],
        "Excluded tooth voxels A (%)": au["excluded_tooth_a_pct"],
    })
seg_stage1_df = pd.DataFrame(seg_rows).set_index("Reply")
seg_stage1_df.to_csv(OUT_ROOT / "dentvoxel_seg_stage1_frozen.csv")
display(seg_stage1_df.round(4))

## Step 5 · `dentvoxel_seg` Stages 2 & 3 — Converged 4-Arm 3D 82-Param Learning Curve (`K = 1..60` Scans, 3 Seeds)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.training.runner import plot_segmentation_learning_curve, run_segmentation_three_arm_validation

print("Loading & resampling 60 training 3D CBCT scans to 96x128x128 (cached in RAM ~360 MB)...")
train_seg = load_dentvoxel_seg(DATA_SRC, SPLIT_FILE, split="train", norm_mode="hu_window")
SEG_OUT = OUT_ROOT / "dentvoxel_seg_converged"
seg_runs, seg_summary = run_segmentation_three_arm_validation(
    task="dentvoxel_seg",
    architecture="cbct-wsp82-v1",
    train_samples=train_seg,
    val_samples=val_seg,
    design_lock_path=seg_design,
    responses_lock_path=seg_resp,
    budgets=[1, 2, 4, 8, 16, 32, 60],
    seeds=[0, 1, 2],
    epochs=120,
    lr=3e-2,
    batch_size=4,
    cosine_lr=True,
    device=DEVICE,
    out_dir=SEG_OUT,
    resume=True,
)
display(seg_summary.round(4))
fig = plot_segmentation_learning_curve(seg_summary, task="dentvoxel_seg")
plt.show()
plt.close(fig)
print(f"Saved converged 3D dentvoxel_seg CSVs, representative payloads, and PNG to {SEG_OUT}")

## Step 6 · Visual 3D CBCT Slice Comparison 1 — All 6 Frozen Replies (Axial & Coronal)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.visualization import plot_frozen_replies_grid, select_deterministic_examples

ATTR = "Dataset: DentVoxel 3D CBCT (Figshare, 100 NIfTI scans, 38 structures) · Deterministic validation scan"
VIZ_DIR = OUT_ROOT / "dentvoxel_seg_visuals"
examples = select_deterministic_examples(val_seg, n=2)

for idx, ex in enumerate(examples, start=1):
    for axis in ("axial", "coronal"):
        out_p = VIZ_DIR / f"scan_{idx}_{axis}_frozen_6replies.png"
        plot_frozen_replies_grid(
            ex,
            frozen_seg,
            title=f"DentVoxel Validation Scan {idx} ({axis.title()} Mid-Arch) · All 6 Frozen cbct-wsp82-v1 Replies",
            attribution=ATTR,
            slice_axis=axis,
            out_path=out_p,
        )
        img = plt.imread(out_p)
        plt.figure(figsize=(10, 12)); plt.imshow(img); plt.axis("off"); plt.show()
print(f"Saved 6-reply CBCT overlays to {VIZ_DIR}")

## Step 7 · Visual 3D CBCT Slice Comparison 2 — All 4 Arms Side-by-Side (`K = 60`)

In [ ]:
import json
import matplotlib.pyplot as plt
from wsp_core.visualization import plot_four_arm_comparison

rep_path = OUT_ROOT / "dentvoxel_seg_converged/dentvoxel_seg_representative_payloads.json"
assert rep_path.is_file(), f"Run Step 5 first to generate {rep_path}"
rep_payloads = json.loads(rep_path.read_text(encoding="utf-8"))

for idx, ex in enumerate(examples, start=1):
    for axis in ("axial", "coronal"):
        out_p = VIZ_DIR / f"scan_{idx}_{axis}_four_arms.png"
        plot_four_arm_comparison(
            ex,
            rep_payloads,
            title=f"DentVoxel Validation Scan {idx} ({axis.title()} Mid-Arch) · 4-Arm Comparison (82-Parameter cbct-wsp82-v1)",
            attribution=ATTR,
            slice_axis=axis,
            out_path=out_p,
        )
        img = plt.imread(out_p)
        plt.figure(figsize=(11, 8)); plt.imshow(img); plt.axis("off"); plt.show()
print(f"Saved 4-arm CBCT comparison overlays to {VIZ_DIR}")